# 🌊 IARA: Interface Analysis and Recognition Architecture

> *"Iara is the mother of the waters. In Brazilian folklore, she shapes the rivers and the depths. In structural biology, it is the displacement and structuring of water—the hydrophobic effect—that fundamentally drives proteins to fold, bind, and interact."*

IARA iis a Graph Neural Network (GNN) able to predict the optimal binding sites for *de novo* binder generation with RFdiffusion, BindCraft and BoltzGen.

This notebook allows you to upload any `.pdb` or `.cif` file, predict its interaction hotspots, and instantly visualize the designable pockets in 3D right in your browser.

In [ ]:
# @title 1. Setup Environment & Download Model
# @markdown This cell installs the required Python dependencies and downloads the IARA Neural Network weights.

import os
import sys

print("Installing dependencies (this takes ~1-2 minutes)...")
!pip install -q torch-geometric prody py3Dmol scipy pandas

print("Downloading IARA Model components...")
# URLs point to the official GitHub repository.
github_raw_base = "https://raw.githubusercontent.com/leodeals/IARA/main/scripts_and_MLmodel"
model_url = "https://raw.githubusercontent.com/leodeals/IARA/main/scripts_and_MLmodel/IARA.pth"

if not os.path.exists("predict.py"):
    !wget -q -O predict.py {github_raw_base}/predict.py
if not os.path.exists("IARA.pth"):
    !wget -q -O IARA.pth {model_url}

print("✅ Setup Complete and Model Loaded!")

In [ ]:
# @title 2. Upload Target Structure
# @markdown Run this cell to upload a `.pdb` or `.cif` file from your desktop.

from google.colab import files
import os

print("Upload your target protein structure:")
uploaded = files.upload()

target_file = list(uploaded.keys())[0]
target_name = os.path.splitext(target_file)[0]

print(f"\n✅ Successfully uploaded {target_file}")

In [ ]:
# @title 3. Predict Interaction Hotspots
# @markdown Execute the IARA GNN on your uploaded structure.

!python predict.py --model IARA.pth --input {target_file} --outdir .

scored_pdb = f"{target_name}_IARA.pdb"
if os.path.exists(scored_pdb):
    print(f"\n✨ Prediction completed successfully! Results saved to {scored_pdb}")
else:
    print("\n❌ Prediction failed.")

In [ ]:
# @title 4. Visualize Results & Download
# @markdown This renders your protein in 3D. The surface is colored from **Blue (0% probability)** to **Red (100% probability)**. The bright red patches are high-confidence binding hotspots.

import py3Dmol
from google.colab import files

with open(scored_pdb, 'r') as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, "pdb")

# Set py3Dmol surface coloring based on B-factors (probabilities 0 to 100)
# Py3Dmol uses 'bwr' for Blue -> White -> Red gradient.
view.setStyle({'model': -1}, {'cartoon': {'color': 'white'}})
view.addSurface(py3Dmol.VDW, {
    'opacity': 1.0,
    'colorscheme': {
        'prop': 'b',
        'gradient': 'rwb',
        'min': 0,
        'max': 100
    }
})

view.zoomTo()
view.show()

# Automatically trigger the download of the scored PDB back to the user's computer
files.download(scored_pdb)